<a href="https://colab.research.google.com/github/AlexandreLouzada/exercicios-analise-dados/blob/master/folium_imoveis_GABARITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗺️ Visualização Geoespacial com Folium — Imóveis Baixada Fluminense — GABARITO

**Contesto:** Módulo de inteligência geográfica para mapear oportunidades imobiliárias em **Nova Iguaçu** e **Queimados**, com marcadores, clusters e análise de valor de mercado.

**Colab:** `Arquivo > Fazer upload do notebook`, executar com `Shift + Enter`. `folium` e `folium.plugins` já vêm no Colab. O mapa é renderizado inline (`display(mapa)` ou sem atribuição).

---

## 📥 1. Configuração do Ambiente e Base de Dados

▶️ Execute primeiro — gera 45 imóveis sintéticos com coordenadas geográficas simuladas.

In [1]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)

print("Dataset geográfico gerado:", df_mapa.shape)
df_mapa.head()

Dataset geográfico gerado: (45, 6)


,id_imovel,cidade,valor_venda,tipo,latitude,longitude
0,1,Queimados,613765.60,Casa,-22.714426,-43.571388
1,2,Nova Iguaçu,368197.75,Terreno,-22.737554,-43.439882
2,3,Nova Iguaçu,514047.61,Apartamento,-22.732235,-43.470753
3,4,Nova Iguaçu,532697.20,Terreno,-22.766920,-43.478809
4,5,Queimados,279398.12,Casa,-22.731598,-43.573369


## 🟢 Parte 1 — Inicialização e Marcadores Básicos

### Tarefa 1: Mapa base centralizado na coordenada média, `zoom_start=12`, OpenStreetMap

In [3]:
# Coordenada média de todos os imóveis (centro do mapa)
lat_media = df_mapa['latitude'].mean()
lon_media = df_mapa['longitude'].mean()
print(f"Centro do mapa: ({lat_media:.5f}, {lon_media:.5f})")

# Mapa base OpenStreetMap
mapa_base = folium.Map(
    location=[lat_media, lon_media],
    zoom_start=12,
    tiles='OpenStreetMap'
)
display(mapa_base)

Centro do mapa: (-22.73679, -43.50738)


### Tarefa 2: Marcadores simples nas 5 primeiras linhas com popup (tipo + valor)

In [4]:
mapa_makers = folium.Map(
    location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Iterar sobre as 5 primeiras linhas
for _, linha in df_mapa.head(5).iterrows():
    popup_texto = f"{linha['tipo']} — R$ {linha['valor_venda']:,.2f}"
    folium.Marker(
        location=[linha['latitude'], linha['longitude']],
        popup=popup_texto
    ).add_to(mapa_makers)

display(mapa_makers)

## 🟡 Parte 2 — Customização Visual com Marcadores Circulares

### Tarefa 3–4: `CircleMarker` para todos os imóveis
- Raio fixo de 8 pixels;
- Preenchimento: azul → Nova Iguaçu, laranja → Queimados;
- Tooltip: "Clique para detalhes".

In [5]:
# Cores por cidade
cores = {'Nova Iguaçu': 'blue', 'Queimados': 'orange'}

mapa_circulos = folium.Map(
    location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()],
    zoom_start=12,
    tiles='OpenStreetMap'
)

for _, linha in df_mapa.iterrows():
    folium.CircleMarker(
        location=[linha['latitude'], linha['longitude']],
        radius=8,
        color=cores[linha['cidade']],
        fill=True,
        fill_color=cores[linha['cidade']],
        fill_opacity=0.7,
        tooltip="Clique para detalhes"
    ).add_to(mapa_circulos)

display(mapa_circulos)

## 🔵 Parte 3 — Agrupamento Inteligente (Clustering)

### Tarefas 5–6: `MarkerCluster` + ícones por tipo de imóvel
- `folium.Icon`: verde → Casa, azul → Apartamento, cinza → Terreno.

In [6]:
cores_tipo = {'Casa': 'green', 'Apartamento': 'blue', 'Terreno': 'gray'}

mapa_cluster = folium.Map(
    location=[df_mapa['latitude'].mean(), df_mapa['longitude'].mean()],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Instanciar o cluster e adicionar AO MAPA
cluster = MarkerCluster().add_to(mapa_cluster)

for _, linha in df_mapa.iterrows():
    popup_texto = f"{linha['tipo']} — R$ {linha['valor_venda']:,.2f}"
    folium.Marker(
        location=[linha['latitude'], linha['longitude']],
        popup=popup_texto,
        tooltip="Clique para detalhes",
        icon=folium.Icon(color=cores_tipo[linha['tipo']], prefix='fa', icon='home')
    ).add_to(cluster)  # adicionado ao cluster, não ao mapa

# Tarefa 7: salvar em HTML
mapa_cluster.save('mapa_imoveis_baixada.html')
print("Mapa salvo como 'mapa_imoveis_baixada.html'")
display(mapa_cluster)

Mapa salvo como 'mapa_imoveis_baixada.html'


## 🎓 Leitura de negócio

> Informação extra: verifique em qual cidade há mais ofertas e o valor médio por tipo:

In [7]:
print("Ofertas por cidade:")
print(df_mapa['cidade'].value_counts())
print("\nValor médio por tipo:")
print(df_mapa.groupby('tipo')['valor_venda'].mean().round(2))
print("\nValor médio por cidade:")
print(df_mapa.groupby('cidade')['valor_venda'].mean().round(2))

print("\n💡 Baixe o arquivo: clique no ícone 📁 (ícone de arquivos) e baixe 'mapa_imoveis_baixada.html' — é um mapa interativo standalone.")

Ofertas por cidade:
cidade
Nova Iguaçu    23
Queimados      22
Name: count, dtype: int64

Valor médio por tipo:
tipo
Apartamento    513620.11
Casa           519791.87
Terreno        472611.12
Name: valor_venda, dtype: float64

Valor médio por cidade:
cidade
Nova Iguaçu    529256.90
Queimados      467647.23
Name: valor_venda, dtype: float64

💡 Baixe o arquivo: clique no ícone 📁 (ícone de arquivos) e baixe 'mapa_imoveis_baixada.html' — é um mapa interativo standalone.


### 🚀 Desafio extra
1. Rótulo de valor no popup com `folium.Popup(..., max_width=300)`.
2. Adicione `pt_popups` para Zoom Incrível nos popups.
3. Compare com `folium.Map(tiles='CartoDB positron')`.